# Parte 3 — Experimento A: DistilBERT
### Workshop: Clasificación de Emociones en Twitter

**Modelo:** `distilbert-base-uncased`  
**Pre-entrenamiento:** Wikipedia + BookCorpus (texto formal)  
**Tokenizador:** WordPiece

Este es nuestro **baseline**. Pre-entrenado en texto formal, sin conocimiento específico del lenguaje de Twitter.

**Prerequisito:** haber ejecutado `part-1-data.ipynb` y `part-2-pipeline.ipynb`

In [ ]:
# Ejecutar primero part-2-pipeline.ipynb para tener todas las funciones disponibles
%run part-2-pipeline.ipynb

## Configuración del experimento

In [ ]:
MODEL_CHECKPOINT = "distilbert-base-uncased"
HF_REPO          = "your-username/tweeteval-emotion-distilbert"  # <-- cambia esto
LR               = 2e-5

### 📝 TODO 3.1 — Tokenizar el dataset con DistilBERT

Usa `make_tokenized_dataset` con el tokenizador correcto y guarda el resultado en `ds_distilbert`.

In [ ]:
tok_distilbert = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
ds_distilbert  = make_tokenized_dataset(tok_distilbert)
print(ds_distilbert)


### 📝 TODO 3.2 — Cargar el modelo y contar parámetros

In [ ]:
from transformers import AutoModelForSequenceClassification
model_distilbert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID
)
total = sum(p.numel() for p in model_distilbert.parameters())
trainable = sum(p.numel() for p in model_distilbert.parameters() if p.requires_grad)
print(f"Parámetros totales: {total/1e6:.1f}M | entrenables: {trainable/1e6:.1f}M")


### 📝 TODO 3.3 — Entrenar y evaluar

In [ ]:
trainer_distilbert = make_trainer(
    model_distilbert, tok_distilbert, ds_distilbert,
    output_dir="./checkpoints/distilbert", lr=LR,
)
trainer_distilbert.train()
plot_training_curves(trainer_distilbert, title="DistilBERT")
metrics_distilbert = full_evaluation(trainer_distilbert, ds_distilbert["test"], model_name="DistilBERT")
metrics_distilbert


### Resultados e interpretación — DistilBERT

Tras entrenar (ver salida de la celda anterior o [notebook en Kaggle](https://www.kaggle.com/code/danielrpo1/si7011-tweeteval-emotion-daniel-restrepo)):

| Métrica (test) | Valor aprox. |
|----------------|--------------|
| Accuracy | **69.5%** |
| F1 macro (validación) | **~0.50** |

**Interpretación:** DistilBERT es un baseline sólido pero limitado: fue pre-entrenado en texto formal y tokeniza mal hashtags, menciones y emojis. Confunde a veces **optimism** con **joy** y tiene precision baja en clases minoritarias (warnings de sklearn). Sirve como referencia de “transformer genérico vs modelo específico de Twitter”.

## Push to Hub

In [ ]:
import os
if os.environ.get("HF_TOKEN"):
    trainer_distilbert.push_to_hub(HF_REPO, commit_message="DistilBERT — TweetEval emotion")
    print(f"Modelo en https://huggingface.co/{HF_REPO}")
else:
    trainer_distilbert.save_model("./checkpoints/distilbert/best")
    print("Guardado localmente (sin HF_TOKEN).")
